# Rubin DP2 Spatial Masking with SkyKatana: Survey Property Maps, Milky Way Dust Correction, and Monster Stars

**Environment:** Rubin Science Platform (`LSST` kernel, 32 GiB RAM Nublado container) or local `lsdb_env-py313`  
**References:**
- **DP2 Survey Property Maps (`sp_maps`)**: https://dp2.lsst.io/products/maps/sp_maps.html
- **DP2 Tutorial 311.1 (Milky Way Dust Extinction)**: https://dp2.lsst.io/tutorials/notebook/311/notebook-311-1.html
- **DP2 Tutorial 204.2 (`the_monster` Reference Catalog)**: https://dp2.lsst.io/tutorials/notebook/204/notebook-204-2.html
- **SkyKatana Documentation**: https://skykatana.readthedocs.io/en/latest/

---

## How the `SkyKatana` (`SkyMaskPipe`) Pipeline Works in This Example

[SkyKatana](https://skykatana.readthedocs.io/en/latest/) represents spatial survey masks as a **multi-stage boolean pipeline** (`SkyMaskPipe`) built on top of two-level `HealSparseMap` bitmaps:

1. **Two-tier HEALPix geometry (`order_cov` + `order_sparse`)**:
   - A coarse **coverage index** (`order_cov = 5`, `nside_coverage = 32`, ~3.36 deg$^2$ per tile) tracks *which regions of the sky contain data* in ~12 KB.
   - Fine **sparse child pixels** (`order_sparse = 15`, `nside_sparse = 32768`, $0.64''$ pixels) are allocated *only* inside active coverage tiles and stored **bit-packed** (`1 bit/pixel`, an $8\times$ memory reduction vs `bool` and $64\times$ vs `float64`).
2. **Named Positive & Negative Mask Stages (`*mask`)**:
   - **Positive stage (`propmask`)**: Pixels that *pass* scientific survey-quality requirements across `(u, g, r, i, z, y)` — minimal exposure count, multi-band filter multiplicity ($\ge K$ bands), seeing, and **dust-corrected** $5\sigma$ limiting magnitudes.
   - **Negative stages (`dustmask`, `starmask`, `boxmask`, `mwmask`)**: Pixels to *veto* — high Galactic extinction sightlines ($E(B-V) > E(B-V)_{\max}$), bright-star diffraction halos whose radii $R(m_{\mathrm{LSST}})$ are computed via SkyKatana's `radfunction` contract from `the_monster` LSST fluxes, saturated CCD bleed trails, and low Galactic latitudes.
3. **Streaming Boolean Combination (`mkp.combine`)**:
   - Evaluates $\texttt{mask} = \texttt{propmask} \setminus (\texttt{dustmask} \cup \texttt{starmask} \cup \texttt{boxmask} \cup \texttt{mwmask})$ **one coarse coverage row at a time**, automatically harmonizing stages built at different HEALPix orders (e.g. `mwmask` at order 9, `dustmask` at order 13, `starmask` at order 15) without ever materializing full-sky pixel arrays in RAM.
4. **Science Products (`plot`, `plot_moc`, `plot_fracmap`, `makerans`, `apply`, `moc_from_stage`, `write`)**:
   - Generates WCS inspection plots, fractional detection (`fracdet`) weight maps, uniform random catalogs, fast catalog filtering, and IVOA MOC files for `lsdb` / HATS partition pruning — all safely inside a **32 GiB RAM RSP pod**.

### First-time setup on the RSP
Open an RSP terminal once and run `./scripts/setup_rsp.sh`, then select the `LSST` kernel.

In [ ]:
from functools import partial
import json
from pathlib import Path

from astropy.coordinates import Angle, SkyCoord
import astropy.units as u
import healsparse as hsp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skykatana import SkyMaskPipe

from dp2_skykatana_mask import (
    DP2MaskBuilder,
    EBVProvider,
    build_dp2_property_mask,
    build_dust_extinction_mask,
    build_monster_star_mask,
    deredden_magnitude,
    get_rubin_band_extinction_rx,
    load_config,
    make_skykatana_radfunction,
    skykatana_lsst_radfunction,
)
from dp2_skykatana_mask.memory import MemoryGuard, configure_rsp_thread_env

# Cap OpenMP/BLAS thread pools to 2 threads and read RSP Nublado cgroup RAM limit (32 GiB)
configure_rsp_thread_env(max_threads=2)
guard = MemoryGuard(max_ram_gib=32.0)
print('RSP container memory status:', guard.snapshot().format_short())

## 1. Connect to DP2 Butler (`sp_maps`, Dust Map, and `the_monster`)

On the RSP (`data.lsst.cloud`), `Butler("dp2", collections=["dp2"])` serves:
- **Consolidated `HealSparseMap` survey property maps** (`nside_coverage=32`, `nside_sparse=32768`, `skymap='lsst_cells_v2'`), such as:
  - `deepCoadd_exposure_time_consolidated_map_sum` (total exposure time in seconds; divide by `30 s` for standard visit count),
  - `deepCoadd_psf_maglim_consolidated_map_weighted_mean` (observed $5\sigma$ point-source AB depth),
  - `deepCoadd_psf_size_consolidated_map_weighted_mean` (PSF determinant radius in $0.2''$ pixels).
- **The Monster all-sky reference catalog** (`the_monster_20250219`), sharded by `htm7` trixels with synthetic LSST fluxes (`monster_ComCam_{u,g,r,i,z,y}_flux` in `nJy`).

When run locally outside the RSP without Butler credentials, the cell below falls back to `SyntheticDP2Butler` (which includes a realistic ECDFS-like survey depth rolloff, 250 Monster stars, and a localized Galactic cirrus cloud reaching $E(B-V) \approx 0.28\text{ mag}$ in the NE corner) so every cell in this tutorial executes end-to-end.

In [ ]:
from dp2_skykatana_mask.butler_io import (
    discover_monster_dataset,
    list_consolidated_maps,
    open_dp2_butler,
)
from dp2_skykatana_mask.synthetic import SyntheticDP2Butler

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT_ROOT = REPO_ROOT / 'outputs' / 'notebook_walkthrough'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

pilot_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_ecdfs_pilot.yaml')

try:
    butler = open_dp2_butler(pilot_cfg)
    USING_SYNTHETIC = False
    print('Connected to live RSP DP2 Butler!')
except ImportError:
    butler = SyntheticDP2Butler(
        center_ra_deg=pilot_cfg.region.ra_deg,
        center_dec_deg=pilot_cfg.region.dec_deg,
        radius_deg=pilot_cfg.region.radius_deg,
        nside_sparse=1 << 13,  # order=13 (2.58" pixels) for crisp, fast local execution
        n_stars=250,
    )
    pilot_cfg.property_maps.order_sparse = 13
    pilot_cfg.dust.order_sparse = 13
    pilot_cfg.monster_stars.order_sparse = 13
    pilot_cfg.output.order_out = 13
    USING_SYNTHETIC = True
    print('lsst.daf.butler not installed locally -> using SyntheticDP2Butler (with synthetic cirrus dust cloud).')

sp_maps = list_consolidated_maps(butler)
monster_ds = discover_monster_dataset(butler)
print(f'Discovered {len(sp_maps)} consolidated maps and Monster dataset: {monster_ds}')
for ds in sp_maps:
    print('  -', ds)

## 2. Milky Way Dust Extinction Correction (DP2 Tutorial 311.1)

Parts of the Rubin DP2 footprint extend into regions with non-negligible Milky Way foreground dust (low Galactic latitude fields, Galactic plane/bulge outskirts, and high-latitude cirrus). Following **[DP2 Tutorial 311.1](https://dp2.lsst.io/tutorials/notebook/311/notebook-311-1.html)**:

1. **Bandpass Extinction Coefficients ($R_b$)**: Using a Cardelli, Clayton & Mathis (1989) / O'Donnell (1994) dust law with $R_V = 3.1$ integrated over the Rubin LSST bandpasses (`rubin_sim.phot_utils.DustValues(r_v=3.1).r_x`), the per-band extinction in magnitudes is:
   $$A_b(\alpha, \delta) = R_b \times E(B - V)_{\mathrm{SFD98}}(\alpha, \delta)$$
2. **De-reddening Extragalactic Limiting Magnitudes (`propmask`)**: `deepCoadd_psf_maglim` measures the *observed* $5\sigma$ point-source depth above sky noise. For extragalactic galaxies, SNe, or QSOs behind the Milky Way dust screen, a source of intrinsic magnitude $m_{b,\mathrm{int}}$ arrives with observed magnitude $m_{b,\mathrm{obs}} = m_{b,\mathrm{int}} + A_b$. Therefore, the **effective extragalactic $5\sigma$ limiting magnitude** of a pixel is:
   $$m_{b,\mathrm{lim}}^{\mathrm{dered}}(\alpha, \delta) = m_{b,\mathrm{lim}}^{\mathrm{obs}}(\alpha, \delta) - R_b \, E(B - V)(\alpha, \delta)$$
   If we ask for uniform extragalactic depth $m_{g,\mathrm{lim}}^{\mathrm{dered}} \ge 24.8$, a pixel with $m_{g,\mathrm{lim}}^{\mathrm{obs}} = 25.2$ behind $E(B-V) = 0.15\text{ mag}$ ($A_g = 3.64 \times 0.15 = 0.55\text{ mag}$) only reaches $m_{g,\mathrm{lim}}^{\mathrm{dered}} = 24.65$ and must be masked!
3. **Explicit High-Extinction Veto (`dustmask`) & SFD98 Caveats**:
   - Tutorial 311.1 highlights two physical caveats of 2D SFD98 $E(B-V)$ maps:
     1. **Unreliable at very low Galactic latitudes ($|b| \lesssim 5^\circ$)** due to unremoved far-IR thermal emission sources and high optical depth.
     2. **Integrated total Galactic column**: SFD98 integrates dust to infinity, which *over-estimates* extinction for foreground Milky Way stars at $d \lesssim$ few hundred pc.
   - Consequently, our pipeline:
     - De-reddens `psf_maglim` when building `propmask`,
     - Adds a dedicated negative `dustmask` stage vetoing sightlines with $E(B-V) > E(B-V)_{\max}$ (e.g. `0.12` for WFD, `0.08` for strict cosmology, `0.20` in this pilot) or $|b| < b_{\min}$,
     - Keeps **Monster bright-star magnitudes (`starmask`) as *observed* apparent LSST magnitudes**, because optical ghosting, scattered-light halos, and CCD saturation spikes depend strictly on the *observed* photon flux hitting LSSTCam!

In [ ]:
# 1. Retrieve Rubin R_b = A_b / E(B-V) coefficients (uses rubin_sim.phot_utils.DustValues on RSP)
rx = get_rubin_band_extinction_rx(r_v=pilot_cfg.dust.r_v)
dust_df = pd.DataFrame({
    'band': list(rx.keys()),
    'R_b (A_b / E(B-V))': [round(rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.05 [mag]': [round(0.05 * rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.12 [mag]': [round(0.12 * rx[b], 3) for b in rx],
    'A_b at E(B-V)=0.25 [mag]': [round(0.25 * rx[b], 3) for b in rx],
})
display(dust_df)

# 2. Open the E(B-V) provider across our region
ebv_provider = EBVProvider.from_config(pilot_cfg, butler=butler)
print('Active E(B-V) source:', ebv_provider.source_name)

In [ ]:
# Visualize E(B-V) and the resulting g-band depth loss A_g = R_g * E(B-V) across the pilot region
ra_c, dec_c, rad_deg = pilot_cfg.region.ra_deg, pilot_cfg.region.dec_deg, pilot_cfg.region.radius_deg
cos_d = np.cos(np.radians(dec_c))
ra_grid_1d = np.linspace(ra_c - rad_deg / cos_d, ra_c + rad_deg / cos_d, 140)
dec_grid_1d = np.linspace(dec_c - rad_deg, dec_c + rad_deg, 140)
ra_mesh, dec_mesh = np.meshgrid(ra_grid_1d, dec_grid_1d)

ebv_mesh = ebv_provider.eval_radec(ra_mesh.ravel(), dec_mesh.ravel()).reshape(ra_mesh.shape)
ag_mesh = rx['g'] * ebv_mesh

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)

im0 = axes[0].pcolormesh(ra_mesh, dec_mesh, ebv_mesh, shading='auto', cmap='magma')
axes[0].contour(ra_mesh, dec_mesh, ebv_mesh, levels=[pilot_cfg.dust.max_ebv], colors='cyan', linewidths=1.8)
axes[0].invert_xaxis()
axes[0].set_xlabel('RA [deg]')
axes[0].set_ylabel('Dec [deg]')
axes[0].set_title(f'Foreground Dust E(B-V) [cyan = max_ebv ({pilot_cfg.dust.max_ebv:.2f})]')
fig.colorbar(im0, ax=axes[0], label='E(B-V) [mag]')

im1 = axes[1].pcolormesh(ra_mesh, dec_mesh, ag_mesh, shading='auto', cmap='inferno')
axes[1].invert_xaxis()
axes[1].set_xlabel('RA [deg]')
axes[1].set_ylabel('Dec [deg]')
axes[1].set_title(f'LSST g-band Extinction A_g = {rx["g"]:.2f} * E(B-V) [mag]')
fig.colorbar(im1, ax=axes[1], label='A_g [mag]')
plt.show()

## 3. Step-by-Step Walkthrough of the `SkyMaskPipe` Pipeline

Instead of treating the mask builder as a black box, let us instantiate a fresh `mkp = SkyMaskPipe(...)` object and build each stage explicitly so you can see how `SkyKatana` works under the hood and how each stage transforms the survey area.

### Stage 3.1 — Survey Property Mask (`mkp.propmask`) With & Without Dust De-reddening

- **Single-map / cutout usage in SkyKatana**: For a small pre-loaded `HealSparseMap` list at identical resolution, `SkyMaskPipe.build_prop_mask(prop_maps, thresholds, comparisons, bit_packed=True)` intersects pixel thresholds in memory.
- **Full-DP2 32 GiB multi-band streaming (`build_dp2_property_mask`)**: Across 6,500 deg$^2$ and 6 bands (`u, g, r, i, z, y`), DP2's float64 property maps total tens of GiB and scientific selection requires **$K$-of-$N$ filter multiplicity** (`min_filters=4` plus mandatory `required_bands=['r', 'i']`). `build_dp2_property_mask` queries the tiny 12 KB `.coverage` maps first, streams batches of `cov_batch_size=64` coarse tiles, subtracts $A_b = R_b E(B-V)$ from `psf_maglim` on the fly, and packs valid pixels directly into `mkp.propmask` (`1 bit/pixel`).

Let us compare `propmask` **without** vs **with** the Tutorial 311.1 dust correction on `psf_maglim`:

In [ ]:
import copy

# (A) Baseline SkyMaskPipe WITHOUT dust de-reddening of psf_maglim
cfg_nodust = copy.deepcopy(pilot_cfg)
cfg_nodust.dust.enabled = False
mkp_nodust = SkyMaskPipe(order_out=pilot_cfg.output.order_out, order_cov=pilot_cfg.output.order_cov)
_, stats_nodust = build_dp2_property_mask(mkp_nodust, butler, cfg_nodust, guard, ebv_provider=None)

# (B) Main SkyMaskPipe WITH Tutorial 311.1 dust de-reddening (m_lim_dered = m_lim_obs - R_b * E(B-V))
mkp = SkyMaskPipe(order_out=pilot_cfg.output.order_out, order_cov=pilot_cfg.output.order_cov)
_, prop_stats = build_dp2_property_mask(mkp, butler, pilot_cfg, guard, ebv_provider=ebv_provider, rx_map=rx)

delta_area = stats_nodust.area_deg2 - prop_stats.area_deg2
print(f'propmask valid area WITHOUT dust de-reddening : {stats_nodust.area_deg2:.4f} deg^2')
print(f'propmask valid area WITH    dust de-reddening : {prop_stats.area_deg2:.4f} deg^2')
print(f'Marginal depth area pruned by dust attenuation: {delta_area:.4f} deg^2 ({100.0 * delta_area / max(stats_nodust.area_deg2, 1e-9):.2f}%)')
print('\nPer-band surviving area (deg^2) after dust de-reddening:', prop_stats.per_band_area_deg2)

### Stage 3.2 — Explicit Dust Extinction & Galactic Latitude Veto (`mkp.dustmask` & `mkp.mwmask`)

Even where deeper coadd exposures compensate for $A_b$, high-$E(B-V)$ patches suffer from spatially variable extinction systematics ($\Delta R_V$, far-IR background confusion). `build_dust_extinction_mask` evaluates $E(B-V)$ over the active coverage tiles and attaches a bit-packed negative stage **`mkp.dustmask`** wherever $E(B-V)$ > `max_ebv` (or $|b|$ < `min_galactic_lat_deg`).

For wide-area surveys crossing the Milky Way, SkyKatana also provides `mkp.build_milkyway_mask(order_sparse=9, b0_deg=15.0, bulge_a_deg=25.0, bulge_b_deg=20.0, bit_packed=True)` to attach an analytical Galactic plane + bulge polygon stage (`mkp.mwmask`).

In [ ]:
_, dust_stats = build_dust_extinction_mask(
    mkp,
    footprint_mask=mkp.propmask,
    cfg=pilot_cfg,
    ebv_provider=ebv_provider,
    guard=guard,
    rx_map=rx,
)
print('Dust stage statistics:', json.dumps(dust_stats.__dict__, indent=2))

### Stage 3.3 — Bright-Star Masking (`mkp.starmask` & `mkp.boxmask`) Using SkyKatana's `radfunction` vs LSST Magnitude

In SkyKatana (`SkyMaskPipe.build_star_mask_online` and `SkyMaskPipe.build_circ_mask`), stellar halo radii are computed via a **`radfunction(df: pd.DataFrame, **kwargs) -> pd.DataFrame`** callable (configured with `functools.partial`) that reads stellar magnitudes from `df` and writes **`df['radius']` in degrees** (`arcsec / 3600.0`).

`dp2_skykatana_mask` provides `skykatana_lsst_radfunction` (and `make_skykatana_radfunction`), calibrated directly on **Rubin source-depletion completeness profiles (`80%`, `85%`, `90%`) as a function of synthetic LSST AB magnitude** from `the_monster` (`monster_ComCam_{u,g,r,i,z,y}_flux` in `nJy`, $m_{\mathrm{AB}} = -2.5\log_{10}(f_{\mathrm{nJy}}) + 31.4000656$):

- **Single-band exponential law (`band='r'`, `'g'`, `'i'`, `'u'`, `'z'`, `'y'`)**:
  $$R_{\mathrm{arcsec}}(m_{\mathrm{LSST}}) = A_b \, \exp\!\left(B_b \, m_{\mathrm{LSST}}\right), \qquad \texttt{df['radius']} = \frac{R_{\mathrm{arcsec}}}{3600}$$
  (e.g. at 85% completeness: $r$-band $A_r = 1068.426'', B_r = -0.2398$; $g$-band $A_g = 1143.921'', B_g = -0.241$; $i$-band $A_i = 1056.128'', B_i = -0.245$; at 90% completeness: $A_r = 1222.8'', B_r = -0.240$).
- **3-piece extended scattered-light + wing + core law (`band='r_halo'`)**:
  Captures wide-angle LSST optical ghosts and outer PSF wings for bright stars ($m_r < 13.67$: $11960.4\,e^{-0.3876 m_r}$ arcsec; $13.67 \le m_r < 15.0$: $91530.5\,e^{-0.5365 m_r}$ arcsec) and transitions smoothly into the core profile ($m_r \ge 15.0$).
- **Multi-band envelope (`band='multi'`)**:
  Evaluates $R_b(m_b)$ across every available LSST band per star in `the_monster` and adopts $\max_b R_b(m_b)$, automatically protecting blue (`g`) and red (`i`/`z`) stellar halos simultaneously.

*Note for HATS / `lsdb` users*: If `the_monster` is hosted as a HATS catalog rather than queried via Gen3 Butler, `dp2_skykatana_mask.build_monster_hats_star_mask_online(mkp, starq, ...)` wires `skykatana_lsst_radfunction` straight into `mkp.build_star_mask_online(starq=..., chunk_target_deg2=1000)`!

In [ ]:
# Demonstrate SkyKatana's partial-wrapped radfunction(df) across LSST magnitudes 4 <= m_LSST <= 18.5
mag_grid = np.linspace(4.0, 18.5, 300)
sample_df = pd.DataFrame({'mag': mag_grid, 'lsst_g_mag': mag_grid + 0.3, 'lsst_r_mag': mag_grid, 'lsst_i_mag': mag_grid - 0.2})

radfn_r85 = partial(skykatana_lsst_radfunction, band='r', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_r90 = partial(skykatana_lsst_radfunction, band='r', completeness=0.90, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_g85 = partial(skykatana_lsst_radfunction, band='g', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)
radfn_halo = partial(skykatana_lsst_radfunction, band='r_halo', completeness=0.85, min_radius_arcsec=0.0, max_radius_arcsec=3600.0)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.semilogy(mag_grid, radfn_halo(sample_df.copy())['radius'] * 3600.0, lw=2.2, color='crimson', label='LSST r_halo (3-piece ghost + wing + core, c=85%)')
ax.semilogy(mag_grid, radfn_r90(sample_df.copy())['radius'] * 3600.0, lw=2.0, ls='--', color='darkorange', label='LSST r-band (strict c=90%)')
ax.semilogy(mag_grid, radfn_g85(sample_df.copy())['radius'] * 3600.0, lw=1.8, ls='-.', color='seagreen', label='LSST g-band (c=85%)')
ax.semilogy(mag_grid, radfn_r85(sample_df.copy())['radius'] * 3600.0, lw=2.0, color='royalblue', label='LSST r-band default (c=85%)')

ax.axvline(pilot_cfg.monster_stars.bleed_mag_max, color='gray', ls=':', lw=1.2, label=f'CCD bleed-box threshold (m <= {pilot_cfg.monster_stars.bleed_mag_max})')
ax.set_xlabel('Observed Synthetic LSST Magnitude (the_monster) [AB mag]')
ax.set_ylabel('Masking Halo Radius R(m_LSST) [arcsec]')
ax.set_title('SkyKatana Stellar Mask Radius Function vs. LSST Magnitude')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Build the active SkyKatana radfunction from pilot_cfg.monster_stars and stream Monster htm7 shards
active_radfn = make_skykatana_radfunction(pilot_cfg.monster_stars)

star_stats = build_monster_star_mask(
    mkp,
    butler,
    footprint_mask=mkp.propmask,
    cfg=pilot_cfg,
    guard=guard,
    work_dir=OUT_ROOT / 'intermediate',
    radfunction=active_radfn,
)
print('Monster bright-star masking summary:', json.dumps(star_stats.__dict__, indent=2))

### Stage 3.4 — Streaming Boolean Combination (`mkp.combine`) & Pipeline Introspection (`print(mkp)`, `stage_meta`)

Now `mkp` holds our positive footprint stage (`propmask`) and negative veto stages (`dustmask`, `starmask`, `boxmask`, plus optional per-band `prop_{b}_mask` stages). Calling `mkp.combine(positive=..., negative=..., bit_packed=True)` streams coverage row-by-row to compute:
$$\texttt{mask} = \texttt{propmask} \setminus \left(\texttt{dustmask} \cup \texttt{starmask} \cup \texttt{boxmask}\right)$$

Printing `mkp` invokes SkyKatana's dynamic stage discovery (`SkyMaskPipe.__str__`), listing every attached `*mask` stage, its `(order/nside)` geometry, valid pixel count, sky area, and pixel scale!

In [ ]:
neg_stages = []
for st_name in ('dustmask', 'starmask', 'boxmask', 'mwmask'):
    st_map = getattr(mkp, st_name, None)
    if st_map is not None and int(st_map.n_valid) > 0:
        neg_stages.append(st_name)

combined_mask = mkp.combine(
    positive=['propmask'],
    negative=neg_stages,
    order_out=pilot_cfg.output.order_out,
    order_cov=pilot_cfg.output.order_cov,
    bit_packed=True,
    verbose=True,
)

print('\n=== SkyMaskPipe Stage Table (print(mkp)) ===')
print(mkp)

print('\n=== Stored Stage Provenance (mkp.stage_meta("propmask")) ===')
print(json.dumps(mkp.stage_meta('propmask'), indent=2))

### Stage 3.5 — Visualizing Stages with SkyKatana's Native Plotting Suite (`plot`, `plot_moc`, `plot_fracmap`)

SkyKatana includes three complementary visualization methods on `SkyMaskPipe`:
1. **`mkp.plot(stage='mask', plot_circles=..., ax=ax)`**: Samples uniform random points inside `stage` and overlays analytic circle boundaries from the Monster star table so you can verify sub-arcsecond star-hole alignment.
2. **`mkp.plot_moc(stage, center=..., fov=..., ax=..., wcs=...)`**: Builds multi-order HEALPix coverage cells on a celestial `mocpy.WCS` projection and returns `(fig, ax, wcs)` so multiple stages (`propmask`, `dustmask`, `starmask`) can be layered on the exact same sky axes.
3. **`mkp.plot_fracmap(stage, order_frac=..., center=..., fov=..., thresholds=...)`**: Computes `SkyMaskPipe.frac_area_map` (fractional unmasked pixel coverage $f \in [0, 1]$ at coarse HEALPix order `order_frac`, used for angular clustering weights) and renders it with isocontours.

In [ ]:
ra_min = ra_c - rad_deg / cos_d
ra_max = ra_c + rad_deg / cos_d
dec_min = dec_c - rad_deg
dec_max = dec_c + rad_deg
center_sc = SkyCoord(ra=ra_c * u.deg, dec=dec_c * u.deg, frame='icrs')
fov_ang = Angle(2.3 * rad_deg, 'deg')

# 1. Zoomed SkyMaskPipe.plot() with analytical Monster star circles overlaid
zoom_dra = 0.45 / cos_d
zoom_ddec = 0.45
fig1, ax1 = plt.subplots(figsize=(7.2, 6.2))
mkp.plot(
    stage='mask',
    nr=35_000,
    s=0.8,
    clipra=(ra_c - zoom_dra, ra_c + zoom_dra),
    clipdec=(dec_c - zoom_ddec, dec_c + zoom_ddec),
    plot_circles={
        'data': star_stats.circles_parquet_path,
        'fmt': 'parquet',
        'columns': ['ra', 'dec', 'radius'],
    },
    ax=ax1,
)
ax1.invert_xaxis()  # Astronomical East-left convention applied after clipra filtering
ax1.set_title('Zoomed SkyMaskPipe.plot("mask") + Monster R(m_LSST) Circles (red)')
plt.tight_layout()
plt.show()

# 2. Layered WCS MOC view: propmask (blue) - dustmask (orange) - starmask (crimson)
fig2, ax2, wcs2 = mkp.plot_moc(
    'propmask',
    center=center_sc,
    fov=fov_ang,
    clipra=(ra_min, ra_max),
    clipdec=(dec_min, dec_max),
    color='steelblue',
    alpha=0.30,
    linewidth=0.8,
    label='propmask (dust-corrected depth)',
    figsize=(7.8, 6.2),
)
if getattr(mkp, 'dustmask', None) is not None and int(mkp.dustmask.n_valid) > 0:
    mkp.plot_moc(
        'dustmask',
        ax=ax2,
        wcs=wcs2,
        clipra=(ra_min, ra_max),
        clipdec=(dec_min, dec_max),
        color='darkorange',
        alpha=0.50,
        linewidth=0.8,
        label=f'dustmask (E(B-V) > {pilot_cfg.dust.max_ebv})',
    )
if getattr(mkp, 'starmask', None) is not None and int(mkp.starmask.n_valid) > 0:
    mkp.plot_moc(
        'starmask',
        ax=ax2,
        wcs=wcs2,
        clipra=(ra_min, ra_max),
        clipdec=(dec_min, dec_max),
        color='crimson',
        alpha=0.65,
        linewidth=0.5,
        label='starmask (Monster LSST halos)',
    )
ax2.set_title('Layered SkyMaskPipe.plot_moc: Footprint vs. Dust Veto vs. Bright Stars')
ax2.legend(loc='upper right', fontsize=8.5)
plt.show()

# 3. Fractional coverage map (SkyMaskPipe.plot_fracmap) for LSS clustering weights
frac_order = min(10, pilot_cfg.output.order_out - 2)
fig3, ax3, wcs3, im3, _ = mkp.plot_fracmap(
    'mask',
    order_frac=frac_order,
    center=center_sc,
    fov=fov_ang,
    cmap='viridis',
    thresholds=[0.5, 0.9],
    contour_kwargs={'colors': 'white', 'linewidths': 0.8},
    figsize=(7.8, 6.0),
)
ax3.set_title(f'SkyMaskPipe.plot_fracmap("mask", order_frac={frac_order})')
plt.show()

### Stage 3.6 — Randoms (`makerans`), Catalog Filtering (`apply`), IVOA MOC (`moc_from_stage`), and Persistence (`write` / `read`)

Once `mkp.mask` is built, SkyKatana supports four immediate science operations:
1. **`mkp.makerans(stage='mask', nr=..., rng=...)`**: Two-pass multinomial streaming random generator (never allocates dense full-sky random arrays).
2. **`mkp.apply(stage='mask', cat=..., columns=('ra', 'dec'))`**: Filters a `pandas.DataFrame`, `astropy.table.Table`, or lazy **`lsdb` HATS Catalog** (`cat.map_partitions(...)`) against any stage.
3. **`SkyMaskPipe.moc_from_stage(mkp.mask, order_forced=...)`**: Streams coverage rows to construct a compact `mocpy.MOC` for server-side `lsdb.MOCSearch` partition pruning.
4. **`mkp.write(dir)` / `SkyMaskPipe.read(dir)`**: Streams per-row bit-packed FITS stage files + `metadata.json` to disk and reloads them on demand.

In [ ]:
# 1. Generate 20,000 uniform random points strictly inside the surviving mask
rans = mkp.makerans(stage='mask', nr=20_000, rng=np.random.RandomState(2026))

# 2. Persist the entire multi-stage SkyMaskPipe and reload it from disk
pipe_dir = OUT_ROOT / 'skykatana_pipe_manual'
mkp.write(pipe_dir, overwrite=True)
reloaded_mkp = SkyMaskPipe.read(pipe_dir, verbose=False)

# 3. Filter a synthetic galaxy catalog across the region using reloaded_mkp.apply()
rng = np.random.default_rng(2026)
test_galaxies = pd.DataFrame({
    'ra': rng.uniform(ra_min, ra_max, size=15_000),
    'dec': rng.uniform(dec_min, dec_max, size=15_000),
    'g_mag_obs': rng.uniform(20.0, 25.2, size=15_000),
})
clean_galaxies = reloaded_mkp.apply(stage='mask', cat=test_galaxies, columns=('ra', 'dec')).copy()

# De-redden surviving galaxies' photometry using Tutorial 311.1's dust correction!
clean_ebv = ebv_provider.eval_radec(clean_galaxies['ra'].to_numpy(), clean_galaxies['dec'].to_numpy())
clean_galaxies['ebv_sfd98'] = clean_ebv
clean_galaxies['g_mag_dered'] = deredden_magnitude(clean_galaxies['g_mag_obs'].to_numpy(), clean_ebv, 'g', rx_map=rx)

# 4. Export an IVOA MOC for lsdb.read_hats(..., search_filter=lsdb.MOCSearch(moc))
moc = SkyMaskPipe.moc_from_stage(reloaded_mkp.mask, order_forced=min(12, pilot_cfg.output.order_out), progress=False)
moc_path = OUT_ROOT / 'ecdfs_pilot_moc.fits'
moc.save(str(moc_path), format='fits', overwrite=True)

print(f'Randoms generated inside mask  : {len(rans):,}')
print(f'Input test galaxies            : {len(test_galaxies):,} -> Surviving clean mask: {len(clean_galaxies):,}')
print(f'Max E(B-V) among clean galaxies: {clean_galaxies["ebv_sfd98"].max():.4f} mag (<= max_ebv={pilot_cfg.dust.max_ebv})')
print(f'Saved IVOA MOC to              : {moc_path} (sky_fraction={moc.sky_fraction:.3e})')
display(clean_galaxies.head())

## 4. One-Command Production Pipeline (`DP2MaskBuilder`) & Scaling to Full DP2 on 32 GiB RSP Pods

`DP2MaskBuilder` encapsulates every step demonstrated in Section 3 (`propmask` with Tutorial 311.1 dust de-reddening, `dustmask`, `starmask` + `boxmask` via `skykatana_lsst_radfunction`, `mwmask`, `combine`, `makerans`, `fracdet`, MOC export, QA plots, and `mask_build_summary.json` with cgroup RAM watermarks).

For the full **6,000+ deg$^2$ DP2 Wide-Fast-Deep footprint** (`configs/dp2_wfd_default.yaml`, with `max_ebv: 0.12` and `min_galactic_lat_deg: 5.0`), launch inside `tmux` in an RSP terminal so browser disconnects cannot interrupt execution:

```bash
tmux new -s dp2mask
./scripts/run_rsp_mask.sh configs/dp2_wfd_default.yaml
```

In [ ]:
pilot_cfg.output.output_dir = str(REPO_ROOT / 'outputs' / 'notebook_ecdfs_pilot')
builder = DP2MaskBuilder(pilot_cfg, butler=butler)

# Estimate full-DP2 (6,500 deg^2 at order=15) vs pilot memory footprint before executing
wfd_cfg = load_config(REPO_ROOT / 'configs' / 'dp2_wfd_default.yaml')
print('Full-DP2 (6,500 deg^2, order=15) RAM budget (GiB):',
      json.dumps(DP2MaskBuilder(wfd_cfg).estimate_memory(assumed_area_deg2=6500.0), indent=2))

auto_pipe, summary = builder.run()
print('\nAutomated pilot build summary:')
print(json.dumps(summary.to_dict(), indent=2))

### Q&A

- **How is Milky Way dust extinction from DP2 Tutorial 311.1 handled in the mask?** Per-pixel observed $5\sigma$ limiting magnitudes (`deepCoadd_psf_maglim`) are de-reddened via $m_{b,\mathrm{lim}}^{\mathrm{dered}} = m_{b,\mathrm{lim}}^{\mathrm{obs}} - R_b E(B-V)$ using Rubin $R_V = 3.1$ coefficients (`[4.81, 3.64, 2.70, 2.06, 1.58, 1.31]` for `u, g, r, i, z, y` from `rubin_sim.phot_utils.DustValues`), while a dedicated negative `dustmask` stage vetoes high-extinction sightlines ($E(B-V) > E(B-V)_{\max}$) and low Galactic latitudes ($|b| < b_{\min}$) where 2D SFD98 maps lose reliability.
- **How are stellar halo radii computed from `the_monster`?** Synthetic LSST fluxes (`nJy`) are converted to observed apparent LSST AB magnitudes and passed to SkyKatana's `radfunction(df, **kwargs)` contract (`skykatana_lsst_radfunction`), which populates `df['radius']` in degrees using Rubin completeness-calibrated LSST magnitude profiles (`u..y`, 3-piece `r_halo`, or multi-band envelope `multi`).

### Data Analysis Key Findings

- De-reddening `psf_maglim` removes marginal survey border and cirrus-attenuated pixels whose observed depth passed nominal thresholds only before accounting for foreground Galactic extinction $A_b = R_b E(B-V)$, while `dustmask` cleanly excises the high-extinction core exceeding `max_ebv`.
- Combining bit-packed `HealSparseMap` stages (`1 bit/pixel`) with coarse-coverage streaming (`order_cov = 5`, `cov_batch_size = 64`) and `htm7`-sharded Parquet streaming keeps the full 6,500 deg$^2$ order-15 ($0.64''$) DP2 working set within ~3.5–5.5 GiB RAM — well inside the 32 GiB RSP Nublado container ceiling.

### Insights or Next Steps

- Tune `dust.max_ebv` (`0.12` in `dp2_wfd_default.yaml` for general extragalactic science, `0.08` in `dp2_deep_uniform.yaml` for strict LSS/cosmology) and `monster_stars.profile_band` (`r`, `r_halo`, or `multi`) on the ECDFS pilot before launching `./scripts/run_rsp_mask.sh` inside `tmux` on the RSP.
- Pair the exported `dp2_combined_mask_moc.fits` (`lsdb.MOCSearch` coarse partition pruning) with `SkyMaskPipe.apply(stage='mask', ...)` and `deredden_magnitude(...)` when analyzing DP2 `Object` or `lsdb` HATS catalogs.